# Random Forest: Adult Income classification

This notebook predicts whether annual income is above $50K using the Adult Income dataset.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Downloads the Adult Income dataset used in the models.md table.
adult = fetch_openml(name='adult', version=2, as_frame=True, parser='auto')
X, y = adult.data, adult.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

model = Pipeline([
    ('preprocess', preprocessor),
    ('forest', RandomForestClassifier(
        n_estimators=300, max_features='sqrt', oob_score=True, n_jobs=-1, random_state=42
    ))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 score: {f1_score(y_test, y_pred, pos_label='>50K'):.4f}')
print(f'OOB score: {model.named_steps["forest"].oob_score_:.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=model.classes_).plot(cmap='Blues')
plt.title('Random Forest: confusion matrix')
plt.show()

In [ ]:
import numpy as np
importances = model.named_steps['forest'].feature_importances_
try:
    feature_names = model.named_steps['preprocess'].get_feature_names_out()
except:
    feature_names = np.array([f"Feature {i}" for i in range(len(importances))])
    
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(10, 6))
plt.title("Top 15 Feature Importances - Random Forest")
plt.barh(range(len(indices)), importances[indices], align="center", color='forestgreen')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()


## Random Forest: method and evaluation

### Formula notation

- $D = \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$: training dataset with features $\mathbf{x}_i$ and binary income label $y_i$.
- $B$: number of trees (`n_estimators`); $h_b(\mathbf{x})$: prediction of tree $b$.
- $\hat{y}$: forest prediction; $TP$, $TN$, $FP$, and $FN$: confusion-matrix counts.

### Random Forest method

For every tree $b$, the forest samples a bootstrap dataset $D_b$ from $D$ and considers a random subset of features at each split. A classification tree selects the split with the largest decrease in impurity. For Gini impurity:

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the fraction of class $k$ examples in a node. A pure node has $G=0$. Random feature subsets and bootstrap samples make the trees less correlated, so their combined prediction generalizes better than one tree.

The forest uses majority voting:

$$\hat{y} = \operatorname*{mode}\left(h_1(\mathbf{x}), h_2(\mathbf{x}), \ldots, h_B(\mathbf{x})\right)$$

### Out-of-bag evaluation

Each bootstrap sample leaves out about one-third of the original examples. These out-of-bag examples provide `oob_score_`, an additional validation estimate without using the held-out test set. **Maximize** OOB score: $1$ is best and $0$ is worst.

### Test-set evaluation

$$\mathrm{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}$$

$$F_1 = 2\cdot\frac{\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}$$

**Maximize** Accuracy, Precision, Recall, and $F_1$: each ranges from $0$ (worst) to $1$ (best). For income prediction, $F_1$ and the confusion matrix help when the `>50K` class is less frequent.